In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
import requests
from bs4 import BeautifulSoup

all_books = []  # container for all scraped books

for i in range(1, 16):  # scrape first 5 pages
    if i == 1:
        url = "https://books.toscrape.com/"
    else:
        url = f"https://books.toscrape.com/catalogue/page-{i}.html"

    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        # safely extract each field
        title_tag = book.find("h3").find("a")
        title = title_tag["title"] if title_tag else None

        price_tag = book.find("p", class_="price_color")
        price = price_tag.text if price_tag else None

        rating_tag = book.find("p", class_="star-rating")
        star_rating = rating_tag["class"][1] if rating_tag and len(rating_tag["class"]) > 1 else None

        availability_tag = book.find("p", class_="instock availability")
        availability = availability_tag.text.strip() if availability_tag else "Not available"

        category_tag = soup.find("ul", class_="breadcrumb")
        category = category_tag.find_all("li")[-1].text.strip() if category_tag else None

        book_data = {
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        }

        all_books.append(book_data)

    print(f"Page {i}: {len(books)} books scraped")

print(f"Total books collected: {len(all_books)}")


Page 1: 20 books scraped
Page 2: 20 books scraped
Page 3: 20 books scraped
Page 4: 20 books scraped
Page 5: 20 books scraped
Page 6: 20 books scraped
Page 7: 20 books scraped
Page 8: 20 books scraped
Page 9: 20 books scraped
Page 10: 20 books scraped
Page 11: 20 books scraped
Page 12: 20 books scraped
Page 13: 20 books scraped
Page 14: 20 books scraped
Page 15: 20 books scraped
Total books collected: 300


In [3]:
for book in all_books:
    raw = book.get("price")  # safely get the price field
    if isinstance(raw, str):
        try:
            # remove pound sign, commas, and whitespace
            clean_str = raw.replace('Â£', '').replace(',', '').strip()
            book["price_gbp"] = float(clean_str)
        except ValueError:
            book["price_gbp"] = None  # mark as missing if conversion fails
    else:
        book["price_gbp"] = None



In [4]:
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

for book in all_books:
    raw_rating = book.get("star_rating")
    if isinstance(raw_rating, list) and len(raw_rating) > 1:
        rating_word = raw_rating[1]  # second item is the actual rating word
        book["rating"] = rating_map.get(rating_word, None)
    elif isinstance(raw_rating, str):
        book["rating"] = rating_map.get(raw_rating, None)
    else:
        book["rating"] = None


for book in all_books:
    raw_avail = book.get("availability")
    if isinstance(raw_avail,str):
        book["in_stock"] = "In stock" in raw_avail
    else:
        book["in_stock"] = None



In [5]:
import pandas as pd
df = pd.DataFrame(all_books)
df.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,A Light in the Attic,Â£51.77,Three,In stock,All products,51.77,3,True
1,Tipping the Velvet,Â£53.74,One,In stock,All products,53.74,1,True
2,Soumission,Â£50.10,One,In stock,All products,50.10,1,True
3,Sharp Objects,Â£47.82,Four,In stock,All products,47.82,4,True
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,All products,54.23,5,True


In [6]:
raw_path = "raw_books_data.csv"
df.to_csv(raw_path, index=False)

print(f"Raw data saved successfully at: {raw_path}")

Raw data saved successfully at: raw_books_data.csv


In [8]:
df['rating'] = df['rating'].fillna(df['rating'].median())
df = df.dropna(subset=['price_gbp', 'title', 'category'])
df.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,A Light in the Attic,Â£51.77,Three,In stock,All products,51.77,3,True
1,Tipping the Velvet,Â£53.74,One,In stock,All products,53.74,1,True
2,Soumission,Â£50.10,One,In stock,All products,50.10,1,True
3,Sharp Objects,Â£47.82,Four,In stock,All products,47.82,4,True
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,All products,54.23,5,True


In [9]:
df.drop(columns=['price', 'star_rating', 'availability'], inplace=True)


In [10]:
conversion_rate = 105.5
df["price_inr"] = df['price_gbp'] * conversion_rate
df.head()

,title,category,price_gbp,rating,in_stock,price_inr
0,A Light in the Attic,All products,51.77,3,True,5461.735
1,Tipping the Velvet,All products,53.74,1,True,5669.570
2,Soumission,All products,50.10,1,True,5285.550
3,Sharp Objects,All products,47.82,4,True,5045.010
4,Sapiens: A Brief History of Humankind,All products,54.23,5,True,5721.265


In [11]:
clean_path = "cleaned_books_data.csv"
df.to_csv(clean_path, index=False)

print(f"Cleaned data saved successfully at: {clean_path}")

Cleaned data saved successfully at: cleaned_books_data.csv


In [12]:
# Task 4: Normalized SQLite schema, data insertion, queries, and verification

import sqlite3
import pandas as pd

# Connect to or create the database
# Use a relative path instead of /content/
conn = sqlite3.connect("books_project.db")

# When saving CSVs
df.to_csv("cleaned_books_data.csv", index=False)

cursor = conn.cursor()

# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories(
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books(
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id) REFERENCES categories(category_id)
);
""")

conn.commit()

# Insert categories
categories = df['category'].unique()
for cat in categories:
    cursor.execute("INSERT OR IGNORE INTO categories(category_name) VALUES (?)", (cat,))

# Map category names to IDs
cat_map = {row[1]: row[0] for row in cursor.execute("SELECT * FROM categories").fetchall()}

# Insert books
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO books(title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row['title'], row['price_gbp'], row['price_inr'], row['rating'], int(row['in_stock']), cat_map[row['category']]))

conn.commit()

# Run required SQL queries
print("Query 1: SELECT + WHERE")
q1 = "SELECT title, price_gbp FROM books WHERE rating = 5;"
print(pd.read_sql(q1, conn), "\n")

print("Query 2: ORDER BY + LIMIT")
q2 = "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 10;"
print(pd.read_sql(q2, conn), "\n")

print("Query 3: DISTINCT")
q3 = "SELECT DISTINCT rating FROM books;"
print(pd.read_sql(q3, conn), "\n")

print("Query 4: BETWEEN")
q4 = "SELECT title, rating FROM books WHERE rating BETWEEN 3 AND 5;"
print(pd.read_sql(q4, conn), "\n")

print("Query 5: JOIN")
q5 = """
SELECT b.title, b.rating, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10;
"""
df_join_sql = pd.read_sql(q5, conn)
print(df_join_sql, "\n")

# Verify JOIN equivalence using pandas merge
df_books = pd.read_sql("SELECT * FROM books", conn)
df_cats = pd.read_sql("SELECT * FROM categories", conn)

df_join_merge = pd.merge(df_books, df_cats, on="category_id")[['title', 'rating', 'category_name']]
print("SQL JOIN and pandas merge equivalent:", df_join_sql.equals(df_join_merge))

# Close connection
conn.close()


Query 1: SELECT + WHERE
                                                title  price_gbp
0               Sapiens: A Brief History of Humankind      54.23
1                                         Set Me Free      17.46
2   Scott Pilgrim's Precious Little Life (Scott Pi...      52.29
3                           Rip it Up and Start Again      35.02
4                          Chase Me (Paris Nights #2)      25.27
5                                          Black Dust      34.53
6   Worlds Elsewhere: Journeys Around Shakespeareâ...      40.30
7   The Four Agreements: A Practical Guide to Pers...      17.66
8                                   The Elephant Tree      23.82
9                                      Sophie's World      15.94
10                        Private Paris (Private #10)      47.61
11  #HigherSelfie: Wake Up Your Life. Free Your So...      23.11
12                       We Love You, Charlie Freeman      50.27
13                                             Thirst      17.27
1